# Training `hey_votask.onnx` — the "hey VoTask" wake word

Synthesizes all training speech with Piper TTS, so **you record nothing**.
Budget **2-3 hours** on a free T4 — mostly downloading, not GPU time.

> **This replaces openWakeWord's own notebook**, which as of 2026 cannot run on
> Colab at all: it installs `tensorflow-cpu==2.8.1`, which has no wheel for
> Python 3.12 (what Colab now ships), so pip hard-fails. Several of its data URLs
> are dead too. TensorFlow is only needed for the TFLite export — we want ONNX, so
> dropping it removes the fatal error and most dependency conflicts with it.

> **Not executed end-to-end.** Every fix here is verified against upstream source,
> live URLs and PyPI wheel tags, but nobody has run the whole pipeline start to
> finish. Expect to debug a cell. Risky steps are flagged inline.

**First: Runtime → Change runtime type → GPU (T4).**

## 1. Confirm the GPU and Python version

In [ ]:
!nvidia-smi -L
import sys; print("Python", sys.version)

Expect a T4 and Python 3.12.x. No GPU still works, but takes many hours.

## 2. Install — without TensorFlow

The key deviation from upstream. Note `--no-deps` on openwakeword, and that we
install from **git, not PyPI**: the released package still requires
`tflite-runtime` (no cp312 wheel), while `main` has migrated to
`ai-edge-litert`. That migration was never released to PyPI.

In [ ]:
# openWakeWord from git (PyPI 0.6.0 will not install on Py3.12)
!git clone -q https://github.com/dscripka/openWakeWord.git
!pip install -q -e ./openWakeWord --no-deps

# Deps by hand. NEVER use the [full] extra — it pins protobuf<4, onnx==1.14,
# datasets<3, torchmetrics<1 and will backtrack for 20 minutes before failing.
!pip install -q \
    onnx onnxruntime \
    torch torchaudio torchinfo torchmetrics \
    speechbrain audiomentations torch-audiomentations \
    acoustics pronouncing mutagen soundfile librosa \
    datasets pyyaml tqdm scipy scikit-learn \
    ai-edge-litert speexdsp-ns

# Piper TTS, for synthesizing the positive samples
!pip install -q piper-phonemize-cross piper-tts

`piper-phonemize` now ships a cp312 wheel, so the old "install `-cross` first
to work around the missing 3.12 wheel" advice is obsolete — either works.

## 3. piper-sample-generator, with a layout shim

openWakeWord's `train.py` does `from generate_samples import generate_samples`,
which needs a **top-level** `generate_samples.py`. Upstream moved it into a
`piper_sample_generator/` package, so a fresh clone raises
`ModuleNotFoundError: No module named 'piper'`. This detects the layout and
writes a shim rather than pinning to an old commit.

In [ ]:
import os, pathlib, textwrap

!git clone -q https://github.com/rhasspy/piper-sample-generator.git

root = pathlib.Path("piper-sample-generator")
if not (root / "generate_samples.py").exists():
    inner = root / "piper_sample_generator" / "generate_samples.py"
    assert inner.exists(), f"can't find generate_samples.py; layout changed again: {list(root.iterdir())}"
    (root / "generate_samples.py").write_text(textwrap.dedent("""
        # Shim: train.py expects this module at the top level, upstream moved it
        # into a package. Re-export so both layouts work.
        from piper_sample_generator.generate_samples import *          # noqa: F401,F403
        from piper_sample_generator.generate_samples import generate_samples  # noqa: F401
    """))
    print("wrote top-level shim")
else:
    print("flat layout, no shim needed")

# The TTS voice checkpoint. Only v1.0.0/v2.0.0 carry assets — v3.x releases are empty.
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
print("voice:", os.path.getsize("piper-sample-generator/models/en_US-libritts_r-medium.pt"), "bytes")

## 4. Patch `torch_audiomentations`

It calls `torchaudio.set_audio_backend()`, removed in torchaudio >= 2.1. Crashes on import.

In [ ]:
import importlib.util, pathlib

spec = importlib.util.find_spec("torch_audiomentations")
io_path = pathlib.Path(spec.submodule_search_locations[0]) / "utils" / "io.py"
src = io_path.read_text()

patched = src.replace(
    "torchaudio.set_audio_backend(",
    'getattr(torchaudio, "set_audio_backend", lambda *a, **k: None)(',
)
io_path.write_text(patched)
print("patched:", io_path, "| changed:", src != patched)

Prints `changed: True` the first time. No kernel restart needed — the trainer
runs as a subprocess and re-imports the patched file.

## 5. Training data

- **AudioSet is gone.** Upstream fetches `agkphysics/AudioSet .../bal_train09.tar`,
  now a **404** — that repo moved to Parquet. We use FMA for background noise.
- The ACAV100M feature file is **~17 GB** and dominates the whole runtime. Start
  this cell and go do something else.

In [ ]:
import os, scipy.io.wavfile, numpy as np, datasets, tqdm

# --- room impulse responses (reverb augmentation) ---
os.makedirs("mit_rirs", exist_ok=True)
rir_ds = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                               split="train", streaming=True)
for row in tqdm.tqdm(rir_ds, desc="RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(f"mit_rirs/{name}", 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))

# --- background noise: FMA (AudioSet's tar is 404) ---
os.makedirs("background_clips", exist_ok=True)
fma = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma = iter(fma.cast_column("audio", datasets.Audio(sampling_rate=16000)))
for i in tqdm.tqdm(range(2000), desc="background"):
    row = next(fma)
    scipy.io.wavfile.write(f"background_clips/{i}.wav", 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))

In [ ]:
# --- precomputed negative features (the big one, ~17 GB) ---
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 6. The config

**`target_phrase` is a list, and the variants matter.** "VoTask" is a coined word, so
Piper has no pronunciation for it and phonemises the single token as one mumbled
blob. Splitting it into two words makes the TTS stress both syllables the way people
actually say it. All variants still train one binary classifier.

**`custom_negative_phrases` is not empty here.** "what task", "no task" and "go task"
are what English engines substitute for "vo task" — and they are sentences people
genuinely say in a task manager. Training against them explicitly is what stops the
model waking when someone asks a colleague "what task is next?".

The false-positive knobs are `max_negative_weight` and
`target_false_positives_per_hour` — there is no `target_accuracy` or
`false_activation_penalty` key, whatever circulating guides say.

In [ ]:
import yaml

config = {
    "model_name": "hey_votask",
    "target_phrase": ["hey vo task", "hey voh task", "hey vo tusk", "hey votask"],
    "custom_negative_phrases": [
        "what task", "no task", "go task", "the task", "which task",
        "photo task", "who asked", "vote", "boat", "hey there",
    ],

    # Upstream recommends >=20,000; 100,000+ is better but slower.
    "n_samples": 20000,
    "n_samples_val": 2000,

    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,

    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./my_custom_model",
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./background_clips"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "./validation_set_features.npy",
    "feature_data_files": {
        "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },
    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50,
    },

    "model_type": "dnn",
    "layer_size": 32,
    "steps": 50000,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
}

with open("hey_votask.yaml", "w") as f:
    yaml.dump(config, f)
print(open("hey_votask.yaml").read())

## 7. Generate the positive clips

In [ ]:
!python ./openWakeWord/openwakeword/train.py --training_config hey_votask.yaml --generate_clips

## 8. ⚠️ Resample 22050 → 16000

**The step most likely to bite you, and it fails late.** Piper's libritts voice
outputs **22050 Hz**; the augmentation and feature pipeline assumes **16000 Hz**.
Skip this and you get `Clip does not have the correct sample rate` — or worse, a
model that trains and never fires. Upstream
[issue #296](https://github.com/dscripka/openWakeWord/issues/296), still open.

In [ ]:
import librosa, soundfile as sf, glob, tqdm, os

fixed = 0
for path in tqdm.tqdm(glob.glob("my_custom_model/**/*.wav", recursive=True), desc="resample"):
    y, sr = librosa.load(path, sr=None)
    if sr != 16000:
        sf.write(path, librosa.resample(y, orig_sr=sr, target_sr=16000), 16000)
        fixed += 1
print(f"resampled {fixed} clips to 16 kHz")

# Stale feature arrays must go or they'll be reused at the wrong rate.
for npy in glob.glob("my_custom_model/**/*.npy", recursive=True):
    os.remove(npy)
    print("removed stale", npy)

## 9. Augment, then train

**Do not pass `--convert_to_tflite`** — it needs the TensorFlow stack we
deliberately skipped, and we only want ONNX.

In [ ]:
!python ./openWakeWord/openwakeword/train.py --training_config hey_votask.yaml --augment_clips

In [ ]:
!python ./openWakeWord/openwakeword/train.py --training_config hey_votask.yaml --train_model

## 10. Verify the export matches what the app expects

Worth 10 seconds here rather than discovering a shape mismatch in the browser.
The detector feeds 16 stacked 96-d embeddings and reads one score.

In [ ]:
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession("my_custom_model/hey_votask.onnx")
i, o = sess.get_inputs()[0], sess.get_outputs()[0]
print("input :", i.name, i.shape)    # expect [1, 16, 96]
print("output:", o.name, o.shape)    # expect [1, 1]

score = sess.run(None, {i.name: np.zeros((1, 16, 96), dtype=np.float32)})[0]
print("score on silence:", float(score.ravel()[0]), "(should be near 0)")

If the input isn't `[1,16,96]`, something exported the wrong graph — re-run
cell 9's `--train_model` and make sure no TFLite path ran.

In [ ]:
from google.colab import files
files.download("my_custom_model/hey_votask.onnx")

## Install it in the app

1. Put the file at `client/public/wakeword/hey_votask.onnx`.
2. In `client/.env`, **delete** the two placeholder lines — the code already
   defaults to `/wakeword/hey_votask.onnx`:
   ```diff
   - VITE_WAKEWORD_MODEL_PATH=/wakeword/hey_jarvis_v0.1.onnx
   - VITE_WAKEWORD_PHRASE=hey jarvis
   ```
3. Set `VITE_WAKEWORD_MODE=onnx` to force the on-device path everywhere (leave it
   `auto` if desktop browsers should keep using Web Speech).
4. Restart Vite, or rebuild for Android.

Then tune the threshold on a real device — see
`docs/HEY_VOTASK_TRAINING.md` in the repo for that and for troubleshooting.

**Before shipping commercially:** openWakeWord's *pretrained* models are
CC-BY-NC-SA 4.0, so `hey_jarvis_v0.1.onnx` must not ship — it's a dev placeholder.
Your own trained model should be yours, but `melspectrogram.onnx` and
`embedding_model.onnx` are upstream's frozen feature extractor that every custom
model needs at inference, and we could not get a definitive license answer for
those two. Confirm before a paid release.